<div align="center">

  <h1><img src="https://raw.githubusercontent.com/auxeno/ion/main/assets/logo.png" alt="Ion" width="72"><br>VAE on dSprites</h1>

  [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/auxeno/ion/blob/main/examples/vae_dsprites.ipynb)

</div>

---

Training a variational autoencoder on the [dSprites](https://github.com/google-deepmind/dsprites-dataset) dataset using [Ion](https://github.com/auxeno/ion).

In [ ]:
# !pip install git+https://github.com/auxeno/ion optax plotly

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
import plotly.graph_objects as go
import plotly.io as pio

import ion
from ion import nn

pio.renderers.default = "notebook_connected"

## The dataset

[dSprites](https://github.com/google-deepmind/dsprites-dataset) is a synthetic dataset of 2D shapes (heart, square, ellipse) procedurally generated from 5 independent ground truth factors:

| Factor | Values |
|---|---|
| Shape | 3 (square, ellipse, heart) |
| Scale | 6 levels |
| Orientation | 40 angles |
| X position | 32 steps |
| Y position | 32 steps |

All combinations are present, giving 737,280 binary 64x64 images. Because the factors are known and independent, dSprites is a standard benchmark for evaluating disentangled representation learning.

In [ ]:
from examples._common.datasets import load_dsprites

images = load_dsprites()
print(f"Shape: {images.shape}")
print(f"Dtype: {images.dtype}")

In [ ]:
from plotly.subplots import make_subplots

num_rows, num_cols = 2, 6
img_size = 128

rng = np.random.default_rng(0)
sample_indices = rng.choice(len(images), size=num_rows * num_cols, replace=False)
samples = images[sample_indices, :, :, 0]

fig = make_subplots(rows=num_rows, cols=num_cols, horizontal_spacing=0.02, vertical_spacing=0.02)
for i, img in enumerate(samples):
    row, col = divmod(i, num_cols)
    fig.add_trace(go.Image(z=np.stack([img * 255] * 3, axis=-1)), row=row + 1, col=col + 1)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_layout(
    height=num_rows * img_size,
    width=num_cols * img_size,
    title="dSprites samples",
    margin=dict(l=10, r=10, t=40, b=10),
)
fig.show()

## Building the VAE

A VAE learns a compressed latent representation by training an encoder and decoder jointly. The encoder maps an image to a distribution over latent vectors (parameterized by a mean and log-variance), the decoder reconstructs images from samples of that distribution. The loss combines a reconstruction term (binary cross-entropy, since dSprites images are binary) with a KL divergence term that regularizes the latent space toward a standard normal.

We set `latent_dim=5` to match the 5 ground truth factors in dSprites. If the model learns a disentangled representation, each latent dimension should correspond to one of the underlying factors (shape, scale, orientation, x, y).

In [ ]:
from jaxtyping import Array, Float, PRNGKeyArray

class Encoder(nn.Module):
    conv_1: nn.Conv
    conv_2: nn.Conv
    conv_3: nn.Conv
    dense: nn.Linear
    dense_mean: nn.Linear
    dense_logvar: nn.Linear

    def __init__(self, *, key: PRNGKeyArray) -> None:
        keys = jax.random.split(key, 6)
        self.conv_1 = nn.Conv(1, 16, (4, 4), stride=2, padding=1, key=keys[0])
        self.conv_2 = nn.Conv(16, 32, (4, 4), stride=2, padding=1, key=keys[1])
        self.conv_3 = nn.Conv(32, 64, (4, 4), stride=2, padding=1, key=keys[2])
        self.dense = nn.Linear(4096, 64, key=keys[3])
        self.dense_mean = nn.Linear(64, 5, key=keys[4])
        self.dense_logvar = nn.Linear(64, 5, key=keys[5])

    def __call__(
        self, 
        x: Float[Array, "b 64 64 1"]
    ) -> tuple[Float[Array, "b 5"], Float[Array, "b 5"]]:
        
        x = jax.nn.relu(self.conv_1(x))
        x = jax.nn.relu(self.conv_2(x))
        x = jax.nn.relu(self.conv_3(x))

        x = x.reshape(-1, 4096)

        x = jax.nn.relu(self.dense(x))
        mean = self.dense_mean(x)
        logvar = self.dense_logvar(x)

        return mean, logvar
    
class Decoder(nn.Module):
    dense: nn.Linear
    conv_t_1: nn.ConvTranspose
    conv_t_2: nn.ConvTranspose
    conv_t_3: nn.ConvTranspose

    def __init__(self, *, key: PRNGKeyArray) -> None:
        keys = jax.random.split(key, 4)
        self.dense = nn.Linear(5, 4096, key=keys[0])
        self.conv_t_1 = nn.ConvTranspose(64, 32, (4, 4), stride=2, padding="SAME", key=keys[1])
        self.conv_t_2 = nn.ConvTranspose(32, 16, (4, 4), stride=2, padding="SAME", key=keys[2])
        self.conv_t_3 = nn.ConvTranspose(16, 1, (4, 4), stride=2, padding="SAME", key=keys[3])
    
    def __call__(self, x: Float[Array, "b 5"]) -> Float[Array, "b 64 64 1"]:

        x = jax.nn.relu(self.dense(x))

        x = x.reshape(-1, 8, 8, 64)

        x = jax.nn.relu(self.conv_t_1(x))
        x = jax.nn.relu(self.conv_t_2(x))
        x = self.conv_t_3(x)

        return x

class VAE(nn.Module):
    encoder: Encoder
    decoder: Decoder

    def __init__(self, *, key: PRNGKeyArray) -> None:
        keys = jax.random.split(key)
        self.encoder = Encoder(key=keys[0])
        self.decoder = Decoder(key=keys[1])

    def __call__(
        self, 
        x: Float[Array, "b 64 64 1"], 
        *, 
        key: PRNGKeyArray
    ) -> Float[Array, "b 64 64 1"]:

        mean, logvar = self.encoder(x)
        
        z = self.reparametrize(mean, logvar, key=key)
        
        x = self.decoder(z)
        
        return x

    @staticmethod
    def reparametrize(
        mean: Float[Array, "b 5"], 
        logvar: Float[Array, "b 5"],
        *,  
        key: PRNGKeyArray,
    ) -> Float[Array, "b 5"]:
        
        std = jnp.exp(0.5 * logvar)
        eps = jax.random.normal(shape=mean.shape, key=key)
        z = mean + std * eps

        return z

model = VAE(key=jax.random.key(0))
print("Number of parameters: ", model.num_params)
model

In [ ]:
# Feed in dummy inputs
dummy_input = jnp.ones((5, 64, 64, 1))
model(dummy_input, key=jax.random.key(0)).squeeze()

## Training

The VAE loss is the sum of two terms:

- **Reconstruction loss**: binary cross-entropy between the input and the decoder output, measuring how well the model reconstructs the original image.
- **KL divergence**: regularizes the latent distribution toward a standard normal, preventing the encoder from collapsing to point estimates.

We track both terms separately to monitor training.

In [ ]:
from tqdm import tqdm


def loss_fn(
    model: VAE,
    x: Float[Array, "b 64 64 1"],
    key: PRNGKeyArray,
) -> tuple[Float[Array, ""], tuple[Float[Array, ""], Float[Array, ""]]]:
    """Returns (total_loss, (loss_recon, loss_kl))."""

    mean, logvar = model.encoder(x)
    z = VAE.reparametrize(mean, logvar, key=key)
    logits = model.decoder(z)

    # Binary cross-entropy loss over independent pixels
    bce_pixel = optax.sigmoid_binary_cross_entropy(logits, x)

    # Sum errors per image then average across the batch
    loss_recon = bce_pixel.sum(axis=(1, 2, 3)).mean()

    # KL divergence of each latent vector dimension from the standard normal distribution
    z_kl = -0.5 * (1 + logvar - mean ** 2 - jnp.exp(logvar))

    # Sum for total KL divergence of each vector then average across the batch
    loss_kl = z_kl.sum(axis=-1).mean()

    return loss_recon + loss_kl, (loss_recon, loss_kl)


LEARNING_RATE = 3e-4
BATCH_SIZE = 128
NUM_EPOCHS = 5

optimizer = optax.adam(LEARNING_RATE)

model = VAE(key=jax.random.key(0))
opt_state = optimizer.init(model.params)

num_batches = len(images) // BATCH_SIZE

recon_losses = []
kl_losses = []


@jax.jit
def train_step(model, opt_state, batch, key):
    (loss, (loss_recon, loss_kl)), grads = ion.value_and_grad(loss_fn, has_aux=True)(model, batch, key)
    updates, opt_state = optimizer.update(grads, opt_state)
    model = ion.apply_updates(model, updates)
    return model, opt_state, loss_recon, loss_kl


for epoch in range(NUM_EPOCHS):
    key = jax.random.key(epoch)
    indices = jax.random.permutation(key, len(images))

    epoch_recon, epoch_kl = 0.0, 0.0
    for i in tqdm(range(num_batches), desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}"):
        batch_indices = indices[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
        batch = jnp.asarray(images[batch_indices], dtype=jnp.float32)

        step_key = jax.random.fold_in(key, i)
        model, opt_state, loss_recon, loss_kl = train_step(model, opt_state, batch, step_key)

        epoch_recon += loss_recon.item()
        epoch_kl += loss_kl.item()

    recon_losses.append(epoch_recon / num_batches)
    kl_losses.append(epoch_kl / num_batches)
    print(f"  recon: {recon_losses[-1]:.2f}  kl: {kl_losses[-1]:.2f}")

In [ ]:
total_losses = [r + k for r, k in zip(recon_losses, kl_losses)]

fig = go.Figure()
fig.add_trace(go.Scatter(y=total_losses, mode="lines", name="Total", line=dict(width=5, color="#636EFA"), opacity=0.8))
fig.add_trace(go.Scatter(y=recon_losses, mode="lines", name="Reconstruction", line=dict(width=5, color="#EF553B"), opacity=0.8))
fig.add_trace(go.Scatter(y=kl_losses, mode="lines", name="KL divergence", line=dict(width=5, color="#00CC96"), opacity=0.8))
fig.update_layout(title="Training loss", xaxis_title="Epoch", yaxis_title="Loss", height=400, template="plotly_white")
fig.show()

## Exploring the latent space

Now that the VAE is trained, we can explore what each latent dimension has learned. The sliders below control the 5 latent dimensions. Moving a slider changes one dimension of the latent vector, which is fed through the decoder to generate an image. If the model has learned a disentangled representation, each slider should correspond to a distinct visual factor like shape, scale, rotation, or position.

In [ ]:
@jax.jit
def decode(z):
    """Decode a single latent vector to a 64x64 image."""
    logits = model.decoder(z.reshape(1, 5))
    image = jax.nn.sigmoid(logits)[0, :, :, 0]
    return image


# Precompute decoded images for each slider position
latent_range = np.linspace(-2, 2, 21)
num_dims = 5

# decoded_grid[d][s] = image when dim d is at step s, others at 0
decoded_grid = []
for d in range(num_dims):
    dim_images = []
    for val in latent_range:
        z = jnp.zeros(5).at[d].set(val)
        dim_images.append(np.array(decode(z)))
    decoded_grid.append(dim_images)

mid = len(latent_range) // 2
init_img = np.array(decode(jnp.zeros(5)))

fig = go.Figure(data=go.Heatmap(
    z=init_img[::-1],
    colorscale="Gray",
    showscale=False,
))

slider_spacing = 0.2
sliders = []
for d in range(num_dims):
    steps = []
    for s, val in enumerate(latent_range):
        img = decoded_grid[d][s]
        step = dict(
            method="restyle",
            args=[{"z": [img[::-1]]}],
            label=f"{val:.1f}",
        )
        steps.append(step)

    sliders.append(dict(
        active=mid,
        steps=steps,
        currentvalue=dict(visible=False),
        len=0.8,
        x=0.1,
        y=-0.02 - d * slider_spacing,
        yanchor="top",
        pad=dict(t=0),
    ))

fig.update_layout(
    sliders=sliders,
    height=700,
    width=600,
    template="plotly_white",
    title="Latent space explorer",
    xaxis=dict(showticklabels=False, domain=[0, 1]),
    yaxis=dict(showticklabels=False, scaleanchor="x", domain=[0.25, 1]),
    margin=dict(l=10, r=10, t=50, b=350),
)
fig.show()